# 07 — Métricas de riesgo

## Objetivo

Analizar estadísticamente los resultados generados por la
simulación Monte Carlo.

Se calcularán:

- Rendimiento promedio.
- Volatilidad.
- Probabilidad de pérdida.
- Percentiles.
- Value at Risk (VaR).
- Expected Shortfall (ES).

Las métricas se encuentran separadas de la lógica de simulación.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd

In [3]:
from src.data.loader import load_csv

from src.risk.metrics import (
    calculate_mean_return,
    calculate_volatility,
    calculate_loss_probability,
    calculate_percentiles,
    calculate_var,
    calculate_expected_shortfall,
)

In [4]:
RESULTS_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "sequential_simulation.csv"
)

simulation_data = load_csv(
    RESULTS_PATH
)

simulation_data.head()

,final_value,final_return
0,97.774180,-0.022258
1,118.401679,0.184017
2,86.328718,-0.136713
3,113.237395,0.132374
4,106.676147,0.066761


In [5]:
final_values = (
    simulation_data["final_value"]
    .to_numpy()
)

final_returns = (
    simulation_data["final_return"]
    .to_numpy()
)

INITIAL_VALUE = 100.0

In [6]:
mean_return = calculate_mean_return(
    final_returns
)

volatility = calculate_volatility(
    final_returns
)

loss_probability = calculate_loss_probability(
    final_values,
    INITIAL_VALUE,
)

print(
    f"Rendimiento promedio: "
    f"{mean_return:.4%}"
)

print(
    f"Volatilidad: "
    f"{volatility:.4%}"
)

print(
    f"Probabilidad de pérdida: "
    f"{loss_probability:.4%}"
)

Rendimiento promedio: 13.0619%
Volatilidad: 19.4849%
Probabilidad de pérdida: 25.7000%


In [7]:
percentiles = calculate_percentiles(
    final_values,
    [
        1,
        5,
        25,
        50,
        75,
        95,
        99,
    ],
)

percentile_table = pd.DataFrame(
    {
        "percentile": percentiles.keys(),
        "final_value": percentiles.values(),
    }
)

percentile_table

,percentile,final_value
0,1,75.038500
1,5,84.096203
2,25,99.489410
3,50,111.247042
4,75,125.064852
5,95,147.951328
6,99,165.746138


In [8]:
var_95 = calculate_var(
    final_returns,
    confidence_level=0.95,
)

es_95 = calculate_expected_shortfall(
    final_returns,
    confidence_level=0.95,
)

print(
    f"VaR 95%: {var_95:.4%}"
)

print(
    f"Expected Shortfall 95%: {es_95:.4%}"
)

VaR 95%: 15.9038%
Expected Shortfall 95%: 20.9370%


In [9]:
risk_summary = pd.DataFrame(
    {
        "metric": [
            "mean_return",
            "volatility",
            "loss_probability",
            "VaR_95",
            "expected_shortfall_95",
        ],
        "value": [
            mean_return,
            volatility,
            loss_probability,
            var_95,
            es_95,
        ],
    }
)

risk_summary

,metric,value
0,mean_return,0.130619
1,volatility,0.194849
2,loss_probability,0.257000
3,VaR_95,0.159038
4,expected_shortfall_95,0.209370


In [10]:
RISK_RESULTS_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "risk_metrics.csv"
)

RISK_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

risk_summary.to_csv(
    RISK_RESULTS_PATH,
    index=False,
)

print(
    RISK_RESULTS_PATH
)

c:\Users\REYES\OneDrive\Desktop\9noSemestre\programacion paralela\analisis riesgo montecarlo\results\tables\risk_metrics.csv
